# Spotify Track Analytics — SQL Solutions & Findings

This notebook executes the Spotify SQL questions against the `spotify` table and derives findings from actual outputs. It includes basic, intermediate and advanced analyses plus data-quality checks.

**Prerequisites:** MySQL 8.0+ and a populated `spotify` table. Set `MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_USER`, `MYSQL_PASSWORD`, and `MYSQL_DATABASE`. Credentials are read from environment variables and are not stored here.

## 1. Connect and validate

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
required=["MYSQL_HOST","MYSQL_USER","MYSQL_PASSWORD","MYSQL_DATABASE"]
missing=[k for k in required if not os.getenv(k)]
if missing:
    raise EnvironmentError("Set MySQL environment variables: "+", ".join(missing))
url=(f"mysql+pymysql://{os.environ['MYSQL_USER']}:{os.environ['MYSQL_PASSWORD']}"
     f"@{os.environ['MYSQL_HOST']}:{os.getenv('MYSQL_PORT','3306')}/{os.environ['MYSQL_DATABASE']}")
engine=create_engine(url,pool_pre_ping=True)
def run_query(sql):
    return pd.read_sql_query(text(sql),engine)
source=run_query("""
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT track) AS distinct_track_names,
       SUM(stream) AS total_streams,
       SUM(views) AS total_views,
       SUM(likes) AS total_likes
FROM spotify
""")
display(source)
if source.empty or int(source.loc[0,"row_count"])==0:
    raise ValueError("spotify table is empty. Load the dataset first.")

## 2. Basic — tracks with more than one billion streams

In [ ]:
q1=run_query("""
SELECT track, artist, stream
FROM spotify
WHERE stream > 1000000000
ORDER BY stream DESC
""")
display(q1)
print(f"Finding: {len(q1):,} rows exceed one billion streams. Check duplicate track/artist rows before treating this as a unique-track count.")

## 3. Basic — album and artist pairs

In [ ]:
q2=run_query("""
SELECT DISTINCT album, artist
FROM spotify
ORDER BY artist, album
""")
display(q2.head(30))
print(f"Finding: {len(q2):,} distinct album/artist pairs are recorded in the table.")

## 4. Basic — comments on licensed tracks

In [ ]:
q3=run_query("""
SELECT SUM(comments) AS total_comments,
       COUNT(*) AS licensed_track_rows
FROM spotify
WHERE licensed = TRUE
""")
display(q3)
print("Finding: total comments are summed across rows marked licensed; this is row-level engagement, not a causal effect of licensing.")

## 5. Basic — single releases

In [ ]:
q4=run_query("""
SELECT track, artist, album
FROM spotify
WHERE LOWER(album_type)='single'
ORDER BY artist, track
""")
display(q4.head(30))
print(f"Finding: {len(q4):,} rows are labelled single releases.")

## 6. Basic — track rows by artist

In [ ]:
q5=run_query("""
SELECT artist, COUNT(*) AS track_rows,
       COUNT(DISTINCT track) AS distinct_track_names
FROM spotify
GROUP BY artist
ORDER BY track_rows DESC
""")
display(q5.head(20))
if not q5.empty:
    r=q5.iloc[0]
    print(f"Finding: {r['artist']} has the most rows ({int(r['track_rows']):,}); compare distinct track names to assess duplicate rows.")

## 7. Intermediate — average danceability by album

In [ ]:
q6=run_query("""
SELECT album, AVG(danceability) AS avg_danceability,
       COUNT(*) AS track_rows
FROM spotify
GROUP BY album
ORDER BY avg_danceability DESC
""")
display(q6.head(20))
print("Finding: average danceability describes the audio-feature values in each album group; it does not imply audience preference.")

## 8. Intermediate — top five energy values

In [ ]:
q7=run_query("""
SELECT track, artist, energy
FROM spotify
WHERE energy IS NOT NULL
ORDER BY energy DESC
LIMIT 5
""")
display(q7)
print("Finding: these are the five highest energy values in the table; check ties and duplicate tracks before publishing a unique ranking.")

## 9. Intermediate — official-video views and likes

In [ ]:
q8=run_query("""
SELECT track, artist, views, likes
FROM spotify
WHERE official_video = TRUE
ORDER BY views DESC
""")
display(q8.head(30))
print(f"Finding: {len(q8):,} rows are marked as official videos; compare views and likes as engagement indicators, not proof of satisfaction.")

## 10. Intermediate — total views by album

In [ ]:
q9=run_query("""
SELECT album, SUM(views) AS total_views,
       COUNT(*) AS track_rows
FROM spotify
GROUP BY album
ORDER BY total_views DESC
""")
display(q9.head(20))
if not q9.empty:
    r=q9.iloc[0]
    print(f"Finding: {r['album']} has the highest summed views ({r['total_views']:,.0f}) in the recorded data.")

## 11. Intermediate — tracks where Spotify streams exceed YouTube views

In [ ]:
q10=run_query("""
SELECT track, artist, stream, views, stream-views AS difference
FROM spotify
WHERE most_played_on='Spotify'
  AND stream IS NOT NULL AND views IS NOT NULL
  AND stream > views
ORDER BY difference DESC
""")
display(q10.head(30))
print(f"Finding: {len(q10):,} rows meet the platform and stream-vs-view conditions. Streams and views may use different counting definitions and should be compared cautiously.")

## 12. Advanced — top three viewed tracks per artist

In [ ]:
q11=run_query("""
WITH ranked AS (
 SELECT artist, track, views,
        ROW_NUMBER() OVER (PARTITION BY artist ORDER BY views DESC, track) AS rn
 FROM spotify
 WHERE views IS NOT NULL
)
SELECT artist, track, views, rn
FROM ranked
WHERE rn<=3
ORDER BY artist, rn
""")
display(q11.head(60))
print("Finding: the query returns up to three ranked rows per artist; ROW_NUMBER ensures at most three even when views are tied.")

## 13. Advanced — liveness above the dataset average

In [ ]:
q12=run_query("""
SELECT track, artist, liveness
FROM spotify
WHERE liveness > (SELECT AVG(liveness) FROM spotify)
ORDER BY liveness DESC
""")
display(q12.head(30))
print(f"Finding: {len(q12):,} rows have liveness above the table-wide average; this is a relative audio-feature comparison.")

## 14. Advanced — energy range by album

In [ ]:
q13=run_query("""
SELECT album, MAX(energy) AS highest_energy,
       MIN(energy) AS lowest_energy,
       MAX(energy)-MIN(energy) AS energy_range
FROM spotify
WHERE energy IS NOT NULL
GROUP BY album
ORDER BY energy_range DESC
""")
display(q13.head(20))
if not q13.empty:
    r=q13.iloc[0]
    print(f"Finding: {r['album']} has the widest recorded energy range ({r['energy_range']:.3f}); album sizes may differ, so inspect track counts too.")

## 15. Advanced — energy-to-liveness ratio

In [ ]:
q14=run_query("""
SELECT track, artist, energy, liveness,
       energy/NULLIF(liveness,0) AS energy_liveness_ratio
FROM spotify
WHERE liveness IS NOT NULL AND liveness<>0
  AND energy/NULLIF(liveness,0)>1.2
ORDER BY energy_liveness_ratio DESC
""")
display(q14.head(30))
print(f"Finding: {len(q14):,} rows exceed the ratio threshold. The ratio is an exploratory rule, not a validated music-quality score.")

## 16. Advanced — cumulative likes ordered by views

Use a deterministic tie-breaker so the cumulative sum is repeatable. The running total is ordered from fewer to more views.

In [ ]:
q15=run_query("""
SELECT track, artist, views, likes,
       SUM(likes) OVER (
           ORDER BY views, track, artist
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS cumulative_likes
FROM spotify
WHERE views IS NOT NULL
ORDER BY views, track, artist
""")
display(q15.head(30))
print("Finding: cumulative likes depend on row order; this is a running total across rows, not an album or artist total.")

## Executive findings

Use the result tables to report the leading artist/album/track groups and platform differences. Before making a business recommendation, check missing values, duplicated rows, time period, metric definitions and whether Spotify streams and YouTube views are comparable. Do not claim query performance improvement without measured execution plans and timings.